# 05 — Final Scoring and Ranking

Creates the final six-factor priority score and ranks all eligible candidate segments.

Safety densities are capped at the 99th percentile before normalization to reduce the influence of extreme short-segment values.

### Final balanced weights

| Factor | Weight |
|---|---:|
| Crash density | 30% |
| VRU injury density | 25% |
| School access | 15% |
| Subway access | 10% |
| Vision Zero context | 10% |
| Protected-network endpoint proximity | 10% |

Traffic is excluded from the core score because observed traffic coverage is too sparse for a citywide comparison.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project = Path.cwd().resolve()
if project.name == "notebooks":
    project = project.parent

raw = project / "data" / "raw"
interim = project / "data" / "interim"
processed = project / "data" / "processed"
tables_dir = project / "outputs" / "tables"

interim.mkdir(parents=True, exist_ok=True)
processed.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project)

import geopandas as gpd

In [ ]:
candidates = gpd.read_file(
    processed / "bike_candidates_with_metrics_2263.gpkg"
)

candidates["crash_den_mi"] = (
    candidates["crash_count"] / candidates["length_mi"]
)
candidates["vru_injury_den_mi"] = (
    candidates["vru_injured"] / candidates["length_mi"]
)

print(
    candidates[["crash_den_mi", "vru_injury_den_mi"]]
    .quantile([0.90, 0.95, 0.99])
)

In [ ]:
crash_cap = candidates["crash_den_mi"].quantile(0.99)
vru_cap = candidates["vru_injury_den_mi"].quantile(0.99)

candidates["crash_den_capped"] = candidates["crash_den_mi"].clip(
    upper=crash_cap
)
candidates["vru_den_capped"] = candidates["vru_injury_den_mi"].clip(
    upper=vru_cap
)

candidates["crash_norm"] = (
    candidates["crash_den_capped"] /
    candidates["crash_den_capped"].max()
)
candidates["vru_norm"] = (
    candidates["vru_den_capped"] /
    candidates["vru_den_capped"].max()
)
candidates["school_norm"] = (
    candidates["school_count_500ft"] /
    candidates["school_count_500ft"].max()
)
candidates["subway_norm"] = (
    candidates["subway_count_500ft"] /
    candidates["subway_count_500ft"].max()
)

print("Crash 99th-percentile cap:", crash_cap)
print("VRU 99th-percentile cap:", vru_cap)

In [ ]:
candidates["final_score_v2"] = (
    0.30 * candidates["crash_norm"] +
    0.25 * candidates["vru_norm"] +
    0.15 * candidates["school_norm"] +
    0.10 * candidates["subway_norm"] +
    0.10 * candidates["vz_priority"] +
    0.10 * candidates["connectivity_norm"]
)

candidates_ranked_v2 = (
    candidates
    .sort_values(
        [
            "final_score_v2",
            "vru_injury_den_mi",
            "crash_den_mi",
            "school_count_500ft",
            "subway_count_500ft",
            "connectivity_score",
            "SegmentID"
        ],
        ascending=[False, False, False, False, False, False, True]
    )
    .reset_index(drop=True)
)

candidates_ranked_v2["rank_v2"] = np.arange(
    1, len(candidates_ranked_v2) + 1
)

top50_v2 = candidates_ranked_v2.head(50).copy()

print(candidates_ranked_v2["final_score_v2"].describe())
print("Top 50 rows:", len(top50_v2))
print("Unique Top 50 SegmentIDs:", top50_v2["SegmentID"].nunique())

In [ ]:
display_cols = [
    "rank_v2", "SegmentID", "Street", "final_score_v2",
    "crash_den_mi", "vru_injury_den_mi",
    "school_count_500ft", "subway_count_500ft",
    "vz_priority", "connectivity_score"
]
top50_v2[display_cols]

In [ ]:
ranked_out = processed / "bike_candidates_ranked_v2_2263.gpkg"
top50_out = processed / "top50_protected_bike_lane_priority_v2_2263.gpkg"

candidates_ranked_v2.to_file(ranked_out, driver="GPKG")
top50_v2.to_file(top50_out, driver="GPKG")

print("Saved:", ranked_out)
print("Saved:", top50_out)